In [1]:
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from pathlib import Path
from dask.distributed import Client
import sys, os
from datetime import datetime, timedelta

from scores.spatial import fss_2d
from skimage.metrics import structural_similarity

import warnings
warnings.filterwarnings("ignore")

sys.path.append("/home/548/cd3022/repos/nowcast-evaluation/src")
import nowcast_preprocess

In [2]:
client = Client(
    n_workers=24,
    threads_per_worker=1
)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/33865/status,
Dashboard: /proxy/33865/status,Workers: 24
Total threads: 24,Total memory: 95.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:45245,Workers: 0
Dashboard: /proxy/33865/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:40939,Total threads: 1
Dashboard: /proxy/38475/status,Memory: 3.96 GiB
Nanny: tcp://127.0.0.1:40011,


In [3]:
YEAR = "2025"
MONTH = "01"
DAY = "02"

SPATIAL_DIMS = ["latitude", "longitude"]

In [4]:
def preprocess_dit(ds):
    ds = nowcast_preprocess.preprocess_nowcast_dataset(
        ds,
        f_dim_name="lead_time",
        new_f_dim_name="lead_time",
    )
    return ds[["surface_global_irradiance"]]

dit_path = Path(f"/scratch/er8/cd3022/CPDiT/stage2_checkpoints/inference/{YEAR}/{MONTH}/{DAY}")
dit_files = [f for f in dit_path.glob("*.nc")]

dit = xr.open_mfdataset(
    dit_files,
    preprocess = preprocess_dit,
    concat_dim='time',
    combine='nested',
    data_vars='minimal',
    coords='minimal',
    compat='override',
    parallel=True,
    chunks='auto'
)

In [5]:
lat_min = dit.latitude.min().item() + 1
lat_max = dit.latitude.max().item() -1
lon_min = dit.longitude.min().item() + 1
lon_max = dit.longitude.max().item() -1

In [6]:
dit = dit.sel(
    latitude=slice(lat_min, lat_max),
    longitude=slice(lon_min, lon_max)
)

In [7]:
BARRA_CS_PATH = Path("/g/data/ob53/BARRA2/output/reanalysis/AUST-04/BOM/ERA5/historical/hres/BARRA-C2/v1/1hr/rsdscs/latest/")
barra_cs_files = sorted([f for f in BARRA_CS_PATH.glob("*2025*.nc")])
barra_cs = xr.open_mfdataset(barra_cs_files)

barra_cs = barra_cs.interp(
    lat=dit.latitude,
    lon=dit.longitude,
    time=dit.time
)

In [8]:
def preprocess_gso_of(ds):
    ds = nowcast_preprocess.preprocess_nowcast_dataset(
        ds,
        f_dim_name="time",
        new_f_dim_name="lead_time",
        relative_leadtime=True
    )
    return ds.sel(
        latitude=slice(lat_max, lat_min),
        longitude=slice(lon_min, lon_max)
    )[["surface_global_irradiance"]]

gso_of_path = Path(f"/g/data/er8/nowcasts/nwc_20240701_20251231/{YEAR}/{MONTH}/{DAY}")
gso_of_files = [f for f in gso_of_path.rglob("*.nc")]
gso_of = xr.open_mfdataset(
    gso_of_files,
    preprocess = preprocess_gso_of,
    concat_dim='time',
    combine='nested',
    data_vars='minimal',
    coords='minimal',
    compat='override',
    parallel=True,
    chunks='auto'
)


def preprocess_truth(ds):
    return ds.sel(
        latitude=slice(lat_min, lat_max),
        longitude=slice(lon_min, lon_max)
    )[["surface_global_irradiance"]]


truth_path = Path(f"/g/data/rv74/satellite-products/arc/der/himawari-ahi/solar/p1s/latest/{YEAR}/{MONTH}/{DAY}")
truth_files = [f for f in truth_path.rglob("*.nc")]
truth = xr.open_mfdataset(
    truth_files,
    preprocess = preprocess_truth,
    concat_dim='time',
    combine='nested',
    data_vars='minimal',
    coords='minimal',
    compat='override',
    parallel=True,
    chunks='auto'
)

In [12]:
def _ssim_2d(f2d, o2d, data_range, sigma):
    """SSIM for a single (lat, lon) field. NaN if either field has missing data."""
    if np.isnan(f2d).any() or np.isnan(o2d).any():
        return np.nan
    return structural_similarity(
        f2d, o2d,
        data_range=data_range,
        gaussian_weights=True,        # Wang et al. (2004) formulation
        sigma=sigma,
        use_sample_covariance=False,
    )

def ssim_field(f, o, data_range, sigma=1.5):
    return xr.apply_ufunc(
        _ssim_2d, f, o,
        input_core_dims=[["latitude", "longitude"], ["latitude", "longitude"]],
        vectorize=True,               # loops over time (and any other non-core dims)
        dask="parallelized",
        output_dtypes=[float],
        kwargs={"data_range": data_range, "sigma": sigma},
    )

In [13]:
def calc_error_metrics(
    fcst: xr.DataArray,
    obs: xr.DataArray,
    fss_threshold = 500, # W/m2
    fss_window_small = (5, 5), # height * width
    fss_window_large = (100, 100), # height * width
    ssim_data_range = 1200, # W/m2, range of values in image
    ssim_sigma = 1.5,
) -> xr.DataArray:
    """Return RMSE with a single dimension: lead time."""

    # Put obs on exactly the forecast grid (guards against float coordinate mismatch)
    obs = obs.sel({"latitude": fcst["latitude"], "longitude": fcst["longitude"]}, method="nearest")
    obs = obs.assign_coords({"latitude": fcst["latitude"], "longitude": fcst["longitude"]})

    rmse_list = []
    mae_list = []
    fss_sml_list = []
    fss_lrg_list = []
    ssim_list = []

    for lead in fcst["lead_time"].values:
        f = fcst.sel(lead_time=lead)
        valid_times = f["time"].values + lead
        # Observations at each forecast's valid time (NaN where obs are missing)
        o = obs.reindex({"time": valid_times})
        o = o.assign_coords({"time": f["time"].values})

        # needed for FSS
        f.load()
        o.load()
        
        sq_err = (f - o) ** 2
        mse = sq_err.mean(dim=["latitude", "longitude"], skipna=True)
        rmse = np.sqrt(mse)
        rmse = rmse.mean(dim=["time"])
        rmse_list.append(rmse)

        mae = f - o
        mae = mae.mean(dim=["latitude", "longitude", "time"], skipna=True)
        mae_list.append(mae)


        fss_sml = fss_2d(
            f,
            o,
            event_threshold=fss_threshold,
            window_size=fss_window_small,
            spatial_dims=["latitude", "longitude"],
            # preserve_dims=["time"]
        )
        fss_sml_list.append(fss_sml)
        fss_lrg = fss_2d(
            f,
            o,
            event_threshold=fss_threshold,
            window_size=fss_window_large,
            spatial_dims=["latitude", "longitude"],
            # preserve_dims=["time"]
        )
        fss_lrg_list.append(fss_lrg)

        ssim = ssim_field(f, o, data_range=ssim_data_range, sigma=ssim_sigma)
        ssim = ssim.mean(dim=["time"])
        ssim_list.append(ssim)

    rmse_array = xr.concat(rmse_list, dim=fcst["lead_time"]).rename("rmse")
    mae_array = xr.concat(mae_list, dim=fcst["lead_time"]).rename("mae")
    fss_sml_array = xr.concat(fss_sml_list, dim=fcst["lead_time"]).rename("fss_small")
    fss_lrg_array = xr.concat(fss_lrg_list, dim=fcst["lead_time"]).rename("fss_large")
    ssim_array = xr.concat(ssim_list, dim=fcst["lead_time"]).rename("ssim")

    return xr.merge([rmse_array, mae_array, fss_sml_array, fss_lrg_array, ssim_array])

In [14]:
metrics = calc_error_metrics(gso_of["surface_global_irradiance"], truth["surface_global_irradiance"])
metrics = metrics.assign_coords(
    lead_time=metrics.lead_time / np.timedelta64(1, "m")
)

In [ ]:
metrics

In [ ]:
metrics_dit = calc_error_metrics(dit["surface_global_irradiance"], truth["surface_global_irradiance"])
metrics_dit = metrics_dit.assign_coords(
    lead_time=metrics_dit.lead_time / np.timedelta64(1, "m")
)

In [ ]:
metrics.fss_small.plot(label="GSO_OF_small")
metrics_dit.fss_small.plot(label="DiT_small")
metrics.fss_large.plot(label="GSO_OF_large")
metrics_dit.fss_large.plot(label="DiT_large")
plt.legend()

In [ ]:
metrics.ssim.plot(label="GSO_OF")
metrics_dit.ssim.plot(label="DiT")

In [ ]:
metrics.rmse.plot(label="DiT")
metrics_dit.rmse.plot(label="DiT")